# 08 · Pitfalls & troubleshooting

The lessons above used small Monte-Carlo sizes for speed. Here is what to
watch for in real runs.

## 1. No staircase (smooth ramp instead)

Vertex selection needs a simplex LP solver. If none is installed the flux
falls back to the smooth interior solution. Check:

In [ ]:
import cvxpy as cp
print('installed:', cp.installed_solvers())
print('has a simplex solver:', any(s in cp.installed_solvers()
                                   for s in ('HIGHS', 'GLPK', 'SCIPY')))
# 'SCIPY' is bundled, so this is normally True. `pip install highspy`
# adds the faster HiGHS.

## 2. `GeV` affects OSQP precision

The physical flux is GeV-invariant, but OSQP's *relative* accuracy
degrades as `GeV` grows. Smaller, well-conditioned `GeV` gives a much
smaller χ²/c.

In [ ]:
import warnings; warnings.simplefilter('ignore')
from neutrino_analysis_band import NeutrinoAnalysis
import numpy as np
for GeV in (0.32e16, 1e16, 2e16):
    a = NeutrinoAnalysis(background_scenario='flat', intervals='180',
                         GeV=GeV, solver='osqp', T=3)
    r = a.optimize(a.data_vector)
    d, b, M = a.data_vector, a.Bkg_vector, a.M_matrix
    resid = (d - b) - M @ r.x
    inv = np.where(d > 0, 1 / np.where(d > 0, d, 1), 0)
    hand = a.T * np.sum(resid * resid * inv) / a.c
    print('GeV=%.2e : chi2/c=%.2e (flux[0] phys=%.3e)'
          % (GeV, hand, r.x[0] * (a.cm**2 * a.sec)))

## 3. Edge returns `inf`

`step` too small can't reach a far upper edge (reach ≈ `v0·step^25`).
Use `step=1.5`. A too-small `step` like 1.05 only reaches ~3.4× v0.

## 4. Jagged / non-monotonic bands at large `T`

Narrow bands amplify any numerical error. Use a well-conditioned `GeV`,
a large `n_pseudo_edge` (≥500), and regenerate stale band files.

## 5. Sub-`rel_tol` differences are not physical

Two edges closer than `rel_tol` (their resolution) are *equal*. To claim
a scenario difference, report edge ± uncertainty over a **seed ensemble**
and show the difference exceeds it. For weakly-constrained upper edges,
report a **one-sided limit** instead of chasing precision.